In [13]:
import pandas as pd
import numpy as np



import nltk
from nltk.corpus import stopwords



nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bhard\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bhard\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [17]:
df= pd.read_csv('spam.csv')
df.head(10)


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
5,spam,FreeMsg Hey there darling it's been 3 week's n...,NaN,NaN,NaN
6,ham,Even my brother is not like to speak with me. ...,NaN,NaN,NaN
7,ham,As per your request 'Melle Melle (Oru Minnamin...,NaN,NaN,NaN
8,spam,WINNER!! As a valued network customer you have...,NaN,NaN,NaN
9,spam,Had your mobile 11 months or more? U R entitle...,NaN,NaN,NaN


In [18]:
df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, inplace=True)

In [19]:
# renaming the columns
df.rename(columns={'v1':'target', 'v2':'text'}, inplace=True)
text= df['text']
target= df['target']


In [20]:
target.value_counts()

target
ham     4825
spam     747
Name: count, dtype: int64

Data Preprocessing


In [21]:
target.value_counts()

target
ham     4825
spam     747
Name: count, dtype: int64

In [25]:
from sklearn.preprocessing import LabelEncoder

Encoder= LabelEncoder()    
df['target']= Encoder.fit_transform(df['target'])

df.head

<bound method NDFrame.head of       target                                               text
0          0  Go until jurong point, crazy.. Available only ...
1          0                      Ok lar... Joking wif u oni...
2          1  Free entry in 2 a wkly comp to win FA Cup fina...
3          0  U dun say so early hor... U c already then say...
4          0  Nah I don't think he goes to usf, he lives aro...
...      ...                                                ...
5567       1  This is the 2nd time we have tried 2 contact u...
5568       0              Will Ì_ b going to esplanade fr home?
5569       0  Pity, * was in mood for that. So...any other s...
5570       0  The guy did some bitching but I acted like i'd...
5571       0                         Rofl. Its true to its name

[5572 rows x 2 columns]>

In [26]:
df.duplicated().sum()

np.int64(403)

In [27]:

df.isnull().sum()

target    0
text      0
dtype: int64

In [28]:
df.drop_duplicates(inplace=True)

Feature Engineering

In [30]:
from nltk.stem.porter import PorterStemmer  #for text stemming

import string  #to handle punctuation and special characters

pr=PorterStemmer()


In [ ]:
def transform_text(text):
    text= text.lower()
    text= nltk.word_tokenize(text)

    y= []
    for i in text:
        if i.isalnum():
            y.append(i)
    
    text= y[:]
    y.clear()  # clear the list y to reuse it for the next step...

    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)

    
    #stemming the words
    text= y[:]
    y.clear()
    for i in text:
        y.append(pr.stem(i))
    
    return " ".join(y)

In [35]:
transform_text("Hi, I am Bhardwaj. I am a Data Scientist and I love to work on NLP projects. I have 2 years of experience in this field.")

'hi bhardwaj data scientist love work nlp project 2 year experi field'

In [38]:
df['text'] = df['text'].apply(transform_text)

In [39]:
df.head(5)

,target,text
0,0,go jurong point crazi avail bugi n great world...
1,0,ok lar joke wif u oni
2,1,free entri 2 wkli comp win fa cup final tkt 21...
3,0,u dun say earli hor u c alreadi say
4,0,nah think goe usf live around though


In [40]:
#converting text into numerical values using Tf-idf vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(df['text']).toarray()
y= df['target'].values


Train Test Split


In [41]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, random_state=2)


Model Training

In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [43]:
lr= LogisticRegression()
svc= SVC(kernel='sigmoid', gamma=1.0)
dt= DecisionTreeClassifier()
rf= RandomForestClassifier()
nb= MultinomialNB()
knn= KNeighborsClassifier()
gb= GradientBoostingClassifier()
ab= AdaBoostClassifier()


In [44]:
clfs= {
    "Logistic Regression": lr,
    "Support Vector Machine": svc,
    "Decision Tree": dt,
    "Random Forest": rf,
    "Naive Bayes": nb,
    "K-Nearest Neighbors": knn,
    "Gradient Boosting": gb,
    "AdaBoost": ab
}

Model Evaluation

In [45]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,precision_score

def train_classifier(clf, X_train, y_train, X_test, y_test):
    clf.fit(X_train, y_train)  # training the model
    y_pred= clf.predict(X_test)  # predicting on the test data
    acc= accuracy_score(y_test, y_pred)  # calculating the accuracy
    pre= classification_report(y_test, y_pred)
    prec= precision_score(y_test, y_pred, average='weighted')
    return acc, pre, prec


In [46]:
accuracy_scores= []
precision_scores= []

for name, clf in clfs.items():
    print("**********", name, "**********")
    acc, pre, prec= train_classifier(clf, X_train, y_train, X_test, y_test)
    print("Accuracy: ", acc)
    print("Precision: ", prec)
    print("Classification Report: \n", pre)
    accuracy_scores.append(acc)
    precision_scores.append(prec)

********** Logistic Regression **********
Accuracy:  0.9564796905222437
Precision:  0.9574817949660491
Classification Report: 
               precision    recall  f1-score   support

           0       0.95      1.00      0.98       896
           1       0.98      0.69      0.81       138

    accuracy                           0.96      1034
   macro avg       0.97      0.84      0.89      1034
weighted avg       0.96      0.96      0.95      1034

********** Support Vector Machine **********
Accuracy:  0.9758220502901354
Precision:  0.9758006179927025
Classification Report: 
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       896
           1       0.97      0.84      0.90       138

    accuracy                           0.98      1034
   macro avg       0.98      0.92      0.94      1034
weighted avg       0.98      0.98      0.98      1034

********** Decision Tree **********
Accuracy:  0.9429400386847195
Precision:  0.9427676